# 03 · CEvNS event simulation

            Replaces `sim_script/CEvNS_Sim.ipynb` and the CEvNS-only path of
            `Data_load_reshape.ipynb`.  We

            1. load the CEvNS multiplicity spectrum,
            2. sample events uniformly inside the fiducial volume,
            3. simulate the S2 pattern + pe info,
            4. CNN-reconstruct the position and compute the pattern likelihood + the
               space-time correlation against the muon record.

In [ ]:
# Auto-discover the package even if the notebook is launched from outside the repo.
import sys, os
ROOT = os.path.abspath(os.path.join(os.getcwd(), '..')) if os.path.basename(os.getcwd()) == 'notebooks' else os.getcwd()
if ROOT not in sys.path:
    sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
plt.rcParams['figure.dpi'] = 110

In [ ]:
from relics_de_sim import DESimConfig, Pipeline
            cfg = DESimConfig.from_yaml('configs/cevns_sim.yaml')
            print('cevns_e_spectrum:', cfg.paths.cevns_e_spectrum)

## 1 – Load muons (provides the time range CEvNS samples in)

In [ ]:
from pathlib import Path
            muon_files = [Path(cfg.paths.muon_track_dir) / f'muon_track.{i}.npy' for i in range(2)]
            pipe = Pipeline(cfg, rng=np.random.default_rng(7))
            pipe.load_muon_tracks(muon_files)
            print(f'time range: {pipe.time_range_s:.1f} s')

## 2 – Sample CEvNS events

In [ ]:
try:
                pipe.simulate_cevns()
                for arr in pipe.cevns_points:
                    if len(arr):
                        print(f'n_e={int(arr["num_e"][0])}: {len(arr)} events')
            except ImportError as exc:
                print('CNN inference unavailable:', exc)

## 3 – Quick look at the CEvNS distribution

            We stack the per-bin arrays and plot total area vs reconstructed radius.

In [ ]:
if pipe.cevns_points:
                arr = np.concatenate(pipe.cevns_points)
                area = arr['pe_by_area'].sum(axis=1)
                rec_x = arr['pos_recon']['xd']
                rec_y = arr['pos_recon']['yd']
                rec_r = np.sqrt(rec_x**2 + rec_y**2)
                fig, axes = plt.subplots(1, 2, figsize=(10, 4))
                axes[0].hist(area, bins=80)
                axes[0].set_xlabel('Σ pe_by_area'); axes[0].set_ylabel('count')
                axes[1].scatter(rec_r, np.log(np.maximum(arr['st_cor'], 1e-30)), s=4, alpha=0.4)
                axes[1].set_xlabel('reconstructed r [mm]'); axes[1].set_ylabel('log(st_cor)')
                plt.tight_layout(); plt.show()

## 4 – Save

In [ ]:
pipe.save('outputs/cevns_sim/notebook_demo')